In [15]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.preprocessing import OneHotEncoder

import math



In [16]:
# 1. Load the data from the CSV file into a pandas DataFrame
csv_file_path = '/Users/tvo/Library/CloudStorage/OneDrive-CaliforniaStateUniversity,Sacramento/college-notes/CSC 180/carprices4.csv'

df = pd.read_csv(csv_file_path)
print("Original Data:")
print(df)
print("-" * 30)

Original Data:
                Car Model  Mileage  Sell Price($)  Age(yrs)
0                  BMW X5    69000          18000         6
1                  BMW X5    35000          34000         3
2                  BMW X5    57000          26100         5
3                  BMW X5    22500          40000         2
4                  BMW X5    46000          31500         4
5                 Audi A5    59000          29400         5
6                 Audi A5    52000          32000         5
7                 Audi A5    72000          19300         6
8                 Audi A5    91000          12000         8
9   Mercedez Benz C class    67000          22000         6
10  Mercedez Benz C class    83000          20000         7
11  Mercedez Benz C class    79000          21000         7
12  Mercedez Benz C class    59000          33000         5
------------------------------


In [17]:
# 2. Cleaning Data
# median_bedrooms = math.floor(df.bedrooms.median())
# df.bedrooms = df.bedrooms.fillna(0.0)

# from word2number import w2n

# print("\nCleaning Experience Data:")

# The NaN values are first replaced with the string 'zero'.
# Then, the apply() method iterates through the column, using the w2n.word_to_num function
# to convert all number words (e.g., 'five', 'ten', 'zero') into integers.
# df.experience = df.experience.fillna("zero").apply(w2n.word_to_num)
# median_experience = df.experience.replace(0, np.nan).median()
# df.experience = df.experience.replace(0, median_experience)

# print("\nData after cleaning experience values to nums:")
# print(df)
# print("-" * 30)





In [18]:
# 3 Dummy Encode the Car Types

# Dummy the 'Car Model' and drop the first category
dummies = pd.get_dummies(df["Car Model"], prefix='Car_Model', drop_first=True)

# Merge the dummies with the original DataFrame
df_processed = pd.concat([df.drop('Car Model', axis='columns'), dummies], axis='columns')


In [19]:
# One Hot Encoding 
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

one_hot_encoded_array = ohe.fit_transform(df[['Car Model']])
encoded_df = pd.DataFrame(one_hot_encoded_array, columns=ohe.get_feature_names_out(['Car Model']))
df_processed_ohe = pd.concat([df.drop('Car Model', axis='columns'), encoded_df], axis='columns')

print(df_processed_ohe)

    Mileage  Sell Price($)  Age(yrs)  Car Model_Audi A5  Car Model_BMW X5  \
0     69000          18000         6                0.0               1.0   
1     35000          34000         3                0.0               1.0   
2     57000          26100         5                0.0               1.0   
3     22500          40000         2                0.0               1.0   
4     46000          31500         4                0.0               1.0   
5     59000          29400         5                1.0               0.0   
6     52000          32000         5                1.0               0.0   
7     72000          19300         6                1.0               0.0   
8     91000          12000         8                1.0               0.0   
9     67000          22000         6                0.0               0.0   
10    83000          20000         7                0.0               0.0   
11    79000          21000         7                0.0               0.0   

In [20]:
# 4. Create and training and test data - DUMMY
target = 'Sell Price($)'
features = [col for col in df_processed.columns if col != target]

X = df_processed[features]
y = df_processed[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [21]:
# 4. Create and training and test data - One Hot Encoding
target = 'Sell Price($)'
features = [col for col in df_processed_ohe.columns if col != target]

X = df_processed_ohe[features]
y = df_processed_ohe[target]

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(X, y, test_size=0.2, random_state=42)



In [22]:
# --- 2. Run Scikit-learn's Linear Regression ---
# Scikit-learn expects a 2D array for features, so we reshape x
model = LinearRegression()
model.fit(X_train, y_train)

print("\n--- Scikit-learn Results ---")
for feature, coef in zip(features, model.coef_):
    print(f"  - {feature}: {coef:.2f}")

print(f"\nModel Intercept: {model.intercept_:.2f}")




--- Scikit-learn Results ---
  - Mileage: -0.72
  - Age(yrs): 2882.45
  - Car Model_Audi A5: -3564.45
  - Car Model_BMW X5: 5130.78

Model Intercept: 55358.60


In [23]:
# Ohe Training
model_ohe = LinearRegression()
model_ohe.fit(X_train_ohe, y_train_ohe)

print("\n--- Scikit-learn Results ---")
for feature, coef in zip(features, model_ohe.coef_):
    print(f"  - {feature}: {coef:.2f}")

print(f"\nModel Intercept: {model_ohe.intercept_:.2f}")




--- Scikit-learn Results ---
  - Mileage: -0.72
  - Age(yrs): 2882.45
  - Car Model_Audi A5: -522.11
  - Car Model_BMW X5: -4086.56
  - Car Model_Mercedez Benz C class: 4608.67

Model Intercept: 55880.71


In [24]:
# Verify against Test Data  SCORE. - DUMMY
y_results = model.predict(X_test)
model.score(X_test, y_test)

from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_results)

print(r2)

-6.262600157603594


In [25]:
# Verify against Test Data  SCORE. - Ohe
y_results = model_ohe.predict(X_test_ohe)
model_ohe.score(X_test_ohe, y_test_ohe)

from sklearn.metrics import r2_score
r2 = r2_score(y_results, y_test_ohe)

print(r2)

-0.2162942799259424


In [26]:
# Predict the Price against new data
# print(X)

candidates_to_predict = [
    [45000, 4, 0, 1], 
    [86000, 7, 1, 0], 
]

predict_price_score = model.predict(candidates_to_predict)

print("\Dummy Predictions:")
for i, salary in enumerate(predict_price_score):
    # Get the original data for the print statement
    milage = candidates_to_predict[i][0]
    age = candidates_to_predict[i][1]
    if candidates_to_predict[i][2] == 1:
        car = "BMW"
    elif candidates_to_predict[i][3] == 1:
        car = "Mercadies Benz"
    else:
        car = "Audi"

    print(f"Candidate {i+1} with Milage {str(milage)} Age {str(age)} Years old,  Car {car}: Predicted Price Score = {predict_price_score[i]:,.2f}")



\Dummy Predictions:
Candidate 1 with Milage 45000 Age 4 Years old,  Car Mercadies Benz: Predicted Price Score = 39,515.19
Candidate 2 with Milage 86000 Age 7 Years old,  Car BMW: Predicted Price Score = 9,852.56


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [27]:
# Predict Ohe
candidates_to_predict = [
    [45000, 4,0, 0, 1], 
    [86000, 7,0, 1, 0], 
]

predict_price_score_ohe = model_ohe.predict(candidates_to_predict)
print("\One Hot Encoding Predictions:")
for i, salary in enumerate(predict_price_score_ohe):
    # Get the original data for the print statement
    milage = candidates_to_predict[i][0]
    age = candidates_to_predict[i][1]
    if candidates_to_predict[i][2] == 1:
        car = "BMW"
    elif candidates_to_predict[i][3] == 1:
        car = "Mercadies Benz"
    else:
        car = "Audi"

    print(f"Candidate {i+1} with Milage {str(milage)} Age {str(age)} Years old,  Car {car}: Predicted Price Score = {predict_price_score_ohe[i]:,.2f}")



\One Hot Encoding Predictions:
Candidate 1 with Milage 45000 Age 4 Years old,  Car Audi: Predicted Price Score = 39,515.19
Candidate 2 with Milage 86000 Age 7 Years old,  Car Mercadies Benz: Predicted Price Score = 9,852.56


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [ ]:
import tkinter as tk
from tkinter import ttk

def calculate_price():
    try:
        company = car_model.get()
        age = int(car_age.get())
        mileage = int(car_mileage.get())
        prediction_models = prediction_model.get()
        
        models = {
            "Linear Regression w/ Dummy": model,
            "Linear Regression w/ 1-hot encoding": model_ohe
        }

        selected_model = models.get(prediction_models)
        
        if not selected_model:
            result_label.config(text="Please select a model.")
            return
            
        if not company:
            result_label.config(text="Please select a car model.")
            return

        features = []
        if prediction_models == "Linear Regression w/ Dummy":
            # The model was trained on features in this order:
            # ['Mileage', 'Age(yrs)', 'Car_Model_Audi A5', 'Car_Model_BMW X5']
            is_audi = 1 if company == "Audi" else 0
            is_bmw = 1 if company == "BMW" else 0
            features = [mileage, age, is_audi, is_bmw]
        elif prediction_models == "Linear Regression w/ 1-hot encoding":
            # We assume the OHE model was trained on all car models,
            # and the order of features is mileage, age, and then the car models
            # alphabetically: Audi, BMW, Mercedes.
            is_audi = 1 if company == "Audi" else 0
            is_bmw = 1 if company == "BMW" else 0
            is_mercedes = 1 if company == "Mercedes" else 0
            features = [mileage, age, is_audi, is_bmw, is_mercedes]

        if features:
            price = selected_model.predict([features])[0]
            result_label.config(text=f'Estimated Price: ${price:,.2f}')
        else:
            result_label.config(text=f'Please select a valid model.')

    except ValueError:
        result_label.config(text=f'Please Enter Valid Numbers for age and mileage')
    except Exception as e:
        result_label.config(text=f'An error occurred: {e}')


def reset_fields():
    car_model.set("")
    car_age.delete(0, tk.END)
    car_mileage.delete(0, tk.END)
    prediction_model.set("")
    result_label.config(text="Estimated Price")

#GUI
root = tk.Tk()
root.title("Assignment 8")
root.geometry("400x500")

# Create a main frame
main_frame = tk.Frame(root, padx=10, pady=10)
main_frame.pack(fill=tk.BOTH, expand=True)

#Prediction Model
tk.Label(main_frame, text="Prediction Model: ").grid(row=0, column=0, sticky="w", pady=5)
prediction_model = ttk.Combobox(main_frame, values=["Linear Regression w/ Dummy", "Linear Regression w/ 1-hot encoding"])
prediction_model.grid(row=0, column=1, sticky="ew", pady=5)


#Car Model
tk.Label(main_frame, text="Car Model: ").grid(row=1, column=0, sticky="w", pady=5)
car_model = ttk.Combobox(main_frame, values=["BMW", "Audi", "Mercedes"])
car_model.grid(row=1, column=1, sticky="ew", pady=5)
#Age
tk.Label(main_frame, text="Age (yrs): ").grid(row=2, column=0, sticky="w", pady=5)
car_age = ttk.Entry(main_frame)
car_age.grid(row=2, column=1, sticky="ew", pady=5)

#Mileage
tk.Label(main_frame, text="Mileage (km): ").grid(row=3, column=0, sticky="w", pady=5)
car_mileage = ttk.Entry(main_frame)
car_mileage.grid(row=3, column=1, sticky="ew", pady=5)

#Buttons
button_frame = tk.Frame(main_frame)
button_frame.grid(row=4, column=0, columnspan=2, pady=10)

calc_button = tk.Button(button_frame, text="Calculate Price", command=calculate_price)
calc_button.grid(row=0, column=0, padx=5)

reset_button = tk.Button(button_frame, text="Reset", command=reset_fields)
reset_button.grid(row=0, column=1, padx=5)

#Result
result_label = tk.Label(main_frame, text="Estimated Price: ")
result_label.grid(row=5, column=0, columnspan=2, pady=10)

main_frame.columnconfigure(1, weight=1)

root.mainloop()

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3

: 